In [ ]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

r = requests.get(url)
text = r.text

#Write in file
with open("input.txt", "w", encoding="utf-8") as f:
    f.write(r.text)

In [14]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
#set() gets all unique characters in text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

#最后有65种不同chars


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [24]:
#create a mapping from characters into integers  tokenize手搓 字符级
stoi = { ch:i for i,ch in enumerate(chars)} #为每个字符创建一个映射 a:0 , b:1
itos = { i:ch for i,ch in enumerate(chars)} #与stoi相反
encode = lambda s: [ stoi[c] for c in s] # 例子：s = "hi!" 遍历字符：'h' → 'i' → '!' 查找索引：stoi['h'] → 7, stoi['i'] → 8, stoi['!'] → 27 结果：[7, 8, 27]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode(' '))
print(decode(encode(' ')))

[1]
 


In [25]:
import torch
data  = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [26]:
#split to train and validation sets 9:1
n = int(0.9*len(data))
train_data = data[:n] #90%
val_data = data[n:]

In [28]:
block_size = 8 #训练数据的大小，一个一个chunk地训练
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1] #预测下一位
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
#batch dimension, 每次random train 一部分的数据
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the max context length for prediction?

def get_batch(split):
    data = train_data if split == 'train' else val_data #选择data 是train还是val
    ix = torch.randint(len(data)-block_size,(batch_size,))  #找4个点起始，len(data)-blocksize保证取到完整的片段
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y


xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb) #这个就是input进transformer的东西
print('targets:')
print(yb.shape)
print(yb)

print('----') # 解释

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [ ]:
#Show how bigram works
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): #继承pytorch的基础神经网络class

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size) #(多少个不同的词/每个词用多少维的向量表示)

    def forward(self, idx, targets = None): #idx ->btc 
        logits = self.token_embedding_table(idx) #(B,T,C) B:batchsize T:blocksize C:vocabsize  logits就是embedding后的向量
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C) #在entropy的计算中只能是二维的，所以logits应该是二维的 
            targets = targets.view(B*T) #targets 只需要正确编号来对比，不需要知道有多少个不同词/字符
            loss = F.cross_entropy(logits, targets) #cross entrophy用于检测预测质量

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)按照概率随机选
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb,yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) #learning rate can be higher in small network

In [ ]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) #目的：改变参数让loss变小
    loss.backward() #计算loss对所有模型参数的导数（梯度grad）
    optimizer.step() #按grad的方向走

print(loss.item()) #tensor出来的真实数值，循环100次后的结果

2.36456298828125


In [141]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))




The, wethe theltondave sEXctichoutss y mpt Offin m: agher t eou henimRI serenkirdy, a or
OMI wngndicomy Gould!
Warsmer abepor f in g, hamen w
po slirrcomyo putheuties;

WA hest ns wis t:
O,

ARGUEShinnsomy, t fouthan h lin hFose bu ur, ber
AThen, -benichasw dothiceran,-L:
PS:


ast tOMo te, bLou, todsipor I:
Gour, s sd oraldak:
Abary,
sts s;
I trof pJus
Whugg; y wharener Couthanow t.
Fo.
Anindather;
Mzyetheathier, foKERY:
Becereintubearkng bo d okntu wethinooetod I llt y thend lle y be pe pr E


In [ ]:
#self attention: 查询向量&键向量，点积得到他们之间的“关系”
torch.manual_seed(1337)
B,T,C = 4,8,32 #batch, time, channels
x = torch.randn(B,T,C)

#perform single head of self attention
head_size = 36
#通过linear转换，x知道自己是key还是query
key = nn.Linear(C,head_size,bias=False)#False mean only apply a matrix multiply with fixed weights
query = nn.Linear(C,head_size,bias=False)
value= nn.Linear(C, head_size,bias=False)
k = key(x) #BT16
q = query(x)

#k和q之间的关系
wei = q @ k.transpose(-2,-1) * head_size ** -0.5#BT16 @ B16T --> BTT 只转16和T， the last part is scaling used to control the variance

tril = torch.tril(torch.ones(T,T)) #三角矩阵1和0
wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei, dim = -1)

v= value(x)
out = wei @ v

#out = wei @ x